In [1]:
#Install our library
!pip install selenium
!pip install openpyxl

In [2]:
# Importing library
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import re

In [3]:
# Get project path
project_path = Path.cwd().parent
data_path = project_path / 'data'

df_path = data_path / 'job applications.xlsx'

In [4]:
# loading the data
df = pd.read_excel(df_path)

In [5]:
# Subsetting GMA_A
df_cognitive_ability_a  = df[df['Item'].str.contains(r'^Cognitive Ability A\d+', regex=True)]

In [6]:
def extract_scrambled_text(scramble):
    """
    Extracts the scrambled word/phrase from a given sentence.

    Args:
    scramble (str): The full text containing instructions and the scrambled text.

    Returns:
    str: The extracted scrambled word/phrase.
    """
    # Use regex to find the last word (the scrambled text)
    match = re.search(r"(\b[a-zA-Z]+\b)$", scramble)
    
    if match:
        return match.group(1)
    return None

In [7]:
# Initiate webdriver
options = webdriver.ChromeOptions()
#options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

# Initialize the WebDriver
driver = webdriver.Chrome(options=options)

# Open the website
driver.get("https://www.thewordfinder.com/multiple-word-anagram-solver/")

In [10]:
# Creating result folder
df_result_version= pd.DataFrame(columns=['original_word', 'valid_anagram_list'])

In [11]:
for _, row in tqdm(df_cognitive_ability_a.iterrows(), total=len(df_cognitive_ability_a)):
    
    # Get scrambled text
    text = row['Question']
    scramble = extract_scrambled_text(text)
    
    # Defining the string
    input_string = scramble
    max_retries = 3  # Define maximum retries
    retries = 0
    anagram_results = []

    while not anagram_results and retries < max_retries:
        try:
            # Locate input field and X button
            input_field = driver.find_element(By.ID, "phrase")
            input_field.send_keys(input_string)
            input_field.send_keys(Keys.RETURN) # Clear text button
            
            # Clicking enter
            time.sleep(3)
            
            # Find the first two ul.anagram-list.row elements
            anagram_lists = driver.find_elements(By.CLASS_NAME, "anagram-result") # Get only first 2

            anagram_results = []
            for result in anagram_lists:
                words = [a.text for a in result.find_elements(By.TAG_NAME, "a")]
                if words:  # Ensure it's not empty
                    anagram_results.append(" ".join(words))               
        except Exception as e:
            print(f"Error occurred: {e}")

        retries += 1  # Increment retry counter
        input_field = driver.find_element(By.ID, "phrase")

        input_field.clear()

    # Save results
    temp_df = pd.DataFrame({'original_word': [scramble], 'valid_anagram_list': [anagram_results]})
    df_result_version = pd.concat([df_result_version, temp_df], ignore_index=True)

100%|██████████| 20/20 [01:57<00:00,  5.89s/it]


In [50]:
def reverse_anagram_list(anagram_list):
    reversed_list = [" ".join(word.split()[::-1]) for word in anagram_list]
    return anagram_list + reversed_list

df_result_version["all_valid_anagram_list"] = df_result_version["valid_anagram_list"].apply(reverse_anagram_list)

In [51]:
# Post-processing - screening for two-word phrases
df_result_version['processed_list'] = df_result_version['all_valid_anagram_list'].apply(
    lambda phrases: [phrase for phrase in phrases if len(phrase.split()) == 2]
)

df_result_version = df_result_version[df_result_version['processed_list'].str.len() > 0]

In [52]:
from collections import defaultdict

# Creating ASCII-value for grouping
def ascii_signature(s: str) -> tuple:
    count = [0]*26
    for char in s.lower():  # ensure everything is consistent
        # Only handle alphabetic chars if that's your use case:
        if 'a' <= char <= 'z':
            count[ord(char) - ord('a')] += 1
    return tuple(count)

df_result_version["ASCII_value"] = df_result_version["original_word"].apply(ascii_signature)

In [53]:
# Grouping by ASCII value
grouped_df = df_result_version.groupby("ASCII_value").agg({
    "original_word": list,      # collect all original_word in one group
    "processed_list": "first"   # take the first processed_list in the group
})

grouped_df.rename(columns={
    "original_word": "phrases",
    "processed_list": "processed_list"
}, inplace=True)

grouped_df = grouped_df.reset_index()

In [54]:
grouped_df

,ASCII_value,phrases,processed_list
0,"(0, 0, 0, 2, 1, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, ...",[hrsysdiietd],"[yiddish rest, rest yiddish]"
1,"(0, 0, 1, 2, 2, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, ...",[oreitedcdo],"[editor coed, editor code, decode trio, decode..."
2,"(0, 1, 0, 0, 2, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, ...","[rbgmuee, rmeebug]","[mere bug, beer mug, beer gum, bug mere, mug b..."
3,"(0, 2, 1, 0, 2, 0, 0, 1, 1, 0, 0, 1, 2, 0, 0, ...",[rbhupeiblcmmes],"[membership club, club membership]"
4,"(1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, ...",[ruutemsam],"[museum tar, museum art, museum rat, mature su..."
5,"(1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, ...",[uyahrcril],"[lurch airy, hairy curl, curly hair, curry hai..."
6,"(1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 2, ...",[aoeirjcok],"[cookie jar, erick joao, cairo joke, jar cooki..."
7,"(1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, ...",[gtnutasriotedcy],"[constitute grady, industry cottage, strategy ..."
8,"(1, 0, 1, 2, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, ...",[rrddclohea],"[orchard led, charred old, arched lord, herald..."
9,"(1, 0, 2, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, ...",[rrootnetccntae],"[concentrate rot, contractor teen, connector t..."


In [39]:
# Output path
anagram_path = project_path / 'intermediate_data' / 'anagram_df.csv'

In [40]:
grouped_df.to_csv(anagram_path, index=False)